<a href="https://colab.research.google.com/github/nosadchiy/public/blob/main/IV_wages_tenure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# 1. Install necessary library
!pip install linearmodels

import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
from linearmodels.iv import IV2SLS

# 2. Load the data directly from your Dropbox link
url = "https://www.dropbox.com/scl/fi/3l7kdzz5ytp6guhsx52ni/nlswork.csv?rlkey=cjv9nyjofwe81ge5mqnqg8sf6&dl=1"
df = pd.read_csv(url)



In [10]:
# 3. Data Prep
df['age_sq'] = df['age'] ** 2

# Drop rows with missing values in the variables we are using
vars_to_keep = ['ln_wage', 'age', 'age_sq', 'not_smsa', 'tenure', 'union', 'south', 'idcode', 'year']
df = df.dropna(subset=vars_to_keep)

# Set the multi-index required for panel models (Entity, Time)
df_panel = df.set_index(['idcode', 'year'])

# 4. FIXED EFFECTS (OLS) MODEL
# Stata: xtreg ln_w age c.age#c.age not_smsa tenure, fe
fe_model = PanelOLS.from_formula(
    'ln_wage ~ age + age_sq + not_smsa + tenure + EntityEffects',
    data=df_panel
)
fe_results = fe_model.fit()

# 5. IV FIXED EFFECTS MODEL (Using De-meaning / Within Transformation)
# Stata: xtivreg ln_w age c.age#c.age not_smsa (tenure = union south), fe

# Demean the variables grouped by 'idcode'
cols_to_demean = ['ln_wage', 'age', 'age_sq', 'not_smsa', 'tenure', 'union', 'south']
df_demeaned = df.groupby('idcode')[cols_to_demean].transform(lambda x: x - x.mean())

# Fit IV2SLS on the demeaned data.
# Note: We omit the intercept (-1) because the de-meaned data has a mean of 0.
iv_fe_model = IV2SLS.from_formula(
    'ln_wage ~ -1 + age + age_sq + not_smsa + [tenure ~ union + south]',
    data=df_demeaned
)
iv_fe_results = iv_fe_model.fit()

# 6. Display Results
print("--- STANDARD FIXED EFFECTS (FE) ---")
print(fe_results.summary.tables[1])

print("\n--- IV FIXED EFFECTS (IV-FE) (De-meaned) ---")
print(iv_fe_results.summary.tables[1])

print("\n--- FIRST STAGE REGRESSION ---")
print(iv_fe_results.first_stage)


--- STANDARD FIXED EFFECTS (FE) ---
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
age            0.0317     0.0034     9.2837     0.0000      0.0250      0.0384
age_sq        -0.0004   5.47e-05    -6.4977     0.0000     -0.0005     -0.0002
not_smsa      -0.1001     0.0126    -7.9306     0.0000     -0.1248     -0.0753
tenure         0.0183     0.0008     22.408     0.0000      0.0167      0.0199

--- IV FIXED EFFECTS (IV-FE) (De-meaned) ---
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
age            0.0118     0.0094     1.2558     0.2092     -0.0066      0.0303
age_sq        -0.0012     0.0002    -6.0642     0